# 🧼 FASE 2 – ETL Light

**Proyecto:** Fase 2: Limpieza de Datos (ETL Light)  
**Autor(a):** Magalí J. Cazella Méndez  
**Fecha:** 5/18/2025 
**Organización:** Proyecto personal para análisis automatizado de DataAnalytics
**Licencia:** Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)  
**Firma digital:** 🧠🔐 MCM–AI·ETL–v1.0–DA

---

Esta notebook tiene por objetivo realizar una limpieza estructurada del dataset `nombre_dataset.csv`, basada en los hallazgos de la Fase 1 (EDA). Incluye:

- Eliminación de columnas irrelevantes  
- Corrección de errores de codificación  
- Normalización de texto y tipos de datos  
- Tratamiento de valores atípicos

> ⚠️ Esta notebook está protegida contra uso comercial no autorizado. Si desea reutilizarla, cite adecuadamente a la autora.


## 📥 Carga del Dataset Original

Se importa el archivo CSV con los datos sin procesar. Se recomienda verificar el encoding del archivo en caso de errores y validar que los datos hayan sido correctamente leídos.


In [ ]:
import pandas as pd

# Ruta al dataset original
ruta_dataset = "C:/Users/magal/PROJECTS/DataAnalytics/datasets/HR_Analytics.csv"

# Carga del archivo CSV
df = pd.read_csv(ruta_dataset)

# Vista rápida de filas y columnas
print(f"Dimensiones: {df.shape}")
df.head()


## 🛡️ Copia de Seguridad del Dataset

Se realiza una copia del DataFrame original para conservar una versión sin modificaciones. Esto permite comparar cambios y facilita la depuración si ocurre algún error durante el proceso de ETL.


In [ ]:
df_original = df.copy()


## 🧽 Eliminación de Columnas Irrelevantes

Se eliminan columnas que tienen un solo valor (constantes) o que no aportan valor informativo. Este paso ayuda a reducir la dimensionalidad innecesaria.


In [ ]:
columnas_irrelevantes = ['EmployeeCount', 'Over18', 'StandardHours']
df.drop(columns=columnas_irrelevantes, inplace=True, errors='ignore')


## 🔠 Limpieza de Strings

Se eliminan espacios en blanco al inicio y fin de los textos y se normaliza la capitalización usando formato Title Case. Esto unifica registros escritos de forma inconsistente.


In [ ]:
for col in df.select_dtypes(include='object'):
    df[col] = df[col].astype(str).str.strip().str.title()


## 🧯 Análisis y Tratamiento de Valores Nulos

Se identifican columnas con valores nulos. Si el porcentaje de nulos supera el 30%, se elimina la columna. Las columnas con menos nulos deben evaluarse caso a caso en fases posteriores.


In [ ]:
# Cálculo del porcentaje de nulos por columna
nulos_porcentaje = df.isnull().mean() * 100
nulos_porcentaje = nulos_porcentaje.sort_values(ascending=False)

# Visualización
print(nulos_porcentaje)


In [ ]:
# Eliminar columnas con más del 30% de nulos
columnas_a_eliminar = nulos_porcentaje[nulos_porcentaje > 30].index
df.drop(columns=columnas_a_eliminar, inplace=True)


## 🔄 Conversión de Tipos de Datos

Se convierte cualquier columna categórica (de tipo texto con pocas categorías) al tipo `category` para optimizar el uso de memoria y facilitar futuros análisis.


In [ ]:
for col in df.select_dtypes(include='object'):
    if df[col].nunique() < df.shape[0] * 0.5:
        df[col] = df[col].astype('category')


## 🏷️ Estandarización de Nombres de Columnas

Se transforman los nombres de las columnas a formato `snake_case`, eliminando espacios y convirtiendo todo a minúsculas. Esto facilita el uso posterior en scripts.


In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')


## 💾 Guardado del Dataset Limpio

Se exporta el archivo limpio a formato `.csv` en la ruta del proyecto. Este dataset será utilizado en la siguiente fase (EDA).


In [ ]:
ruta_salida = "C:/Users/magal/PROJECTS/DataAnalytics/ETL_DataSets/HR_Analytics_limpio.csv"
df.to_csv(ruta_salida, index=False)
print(f"Dataset limpio guardado en: {ruta_salida}")


## ✅ Observaciones de ETL Light (dinámicas)

Este bloque analiza el archivo limpio generado por la ETL (`nombreDataset_limpio.csv`), identificando:
- Columnas eliminadas por ser constantes (desaparecidas respecto al original).
- Columnas eliminadas por nulos > 30%.
- Columnas normalizadas tipo `object`.
- Columnas convertidas a tipo `category`.
El resumen se guarda en formato texto para documentar automáticamente cada ejecución de limpieza.


In [ ]:
import pandas as pd
import numpy as np

# ✅ Cargar datasets desde sus rutas correspondientes - copy as a path y pegar dentro de los encomillados
ruta_original = "C:/Users/magal/PROJECTS/DataAnalytics/datasets/HR_Analytics.csv"
ruta_limpio = "C:/Users/magal/PROJECTS/DataAnalytics/ETL_DataSets/HR_Analytics_limpio.csv"

df_original = pd.read_csv(ruta_original)
df_limpio = pd.read_csv(ruta_limpio)

# ✅ Columnas eliminadas (presentes en el original pero no en el limpio)
columnas_eliminadas = list(set(df_original.columns) - set(df_limpio.columns))

# ✅ Columnas con más del 30% de nulos en el original
columnas_nulos = df_original.isnull().mean()
columnas_nulos_mayor_30 = list(columnas_nulos[columnas_nulos > 0.3].index)

# ✅ Columnas tipo object en el limpio (para verificar normalización aplicada)
columnas_object = df_limpio.select_dtypes(include='object').columns

# ✅ Columnas convertidas a categoría en el limpio
columnas_category = df_limpio.select_dtypes(include='category').columns

# ✅ Armado del texto de observaciones reales
observaciones = f"""
## ✅ Observaciones de ETL Light

- Columnas eliminadas por ser constantes: {', '.join(col for col in columnas_eliminadas) if columnas_eliminadas else 'Ninguna'}
- Columnas eliminadas por nulos > 30%: {', '.join(columnas_nulos_mayor_30) if columnas_nulos_mayor_30 else 'Ninguna'}
- Texto normalizado (title case y espacios): aplicado a {len(columnas_object)} columnas tipo object
- Conversión de tipo aplicada: {len(columnas_category)} columnas se transformaron a `category`
- Archivo guardado como: HR_Analytics_limpio.csv
"""

print(observaciones)

# ✅ Timestamp y firma
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
firma = f"\n\n---\nGenerado por Magalí Cazella – ETL Light [{timestamp}]"

# ✅ Guardado en archivo .txt
ruta_md = "C:/Users/magal/PROJECTS/DataAnalytics/ETL_DataSets/Observaciones_ETL_Light.txt"
with open(ruta_md, 'w', encoding='utf-8') as file:
    file.write(observaciones)

